<a href="https://colab.research.google.com/github/sanagahoi/Embedding-tf-pipeline/blob/main/tf_pipeline_for_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Movie reviews are present as individual text file (one file per review) in review folder.

Folder structure looks like this,

reviews

    |__ positive

        |__pos_1.txt

        |__pos_2.txt

        |__pos_3.txt


    |__ negative

        |__neg_1.txt

        |__neg_2.txt

        |__neg_3.txt


You need to read these reviews using tf.data.Dataset and perform following transformations.

1. Read text review and generate a label from folder name. your dataset should have review text and label as a tuple

In [4]:
import tensorflow as tf

In [5]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

### 1. Mount Google Drive

Run the following cell to mount your Google Drive. This will prompt you to authorize Colab to access your Drive files.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import os

base_path = '/content/drive/My Drive/Colab Notebooks/reviews'

print(f"reviews are been taken from : {base_path}")

reviews are been taken from : /content/drive/My Drive/Colab Notebooks/reviews


In [22]:
# Read file name
reviews_ds = tf.data.Dataset.list_files(os.path.join(base_path,   '*/*/'), shuffle=False)
for review in reviews_ds:
    print(review.numpy())

b'/content/drive/My Drive/Colab Notebooks/reviews/negative/blank_neg.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/negative/neg_1.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/negative/neg_2.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/negative/neg_3.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/positive/blank_pos.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/positive/pos_1.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/positive/pos_2.txt'
b'/content/drive/My Drive/Colab Notebooks/reviews/positive/pos_3.txt'


In [25]:
# Read text review from each file and generate a label from folder name
def process_path(file_path):
    # Extract label from the file path
    parts = tf.strings.split(file_path, os.path.sep)
    label = tf.where(tf.equal(parts[-2], 'positive'),1,0) #1 for positive, 0 for negative

    # read the file content
    text = tf.io.read_file(file_path)
    return text, label

# Map the processing function over the dataset
processed_reviews_ds = reviews_ds.map(process_path)

for text , label in processed_reviews_ds.take(5):
  print("Text: " , text.numpy().decode('utf-8'))
  print('Label: ', label.numpy())


Text:  
Label:  0
Text:  Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.

Label:  0
Text:  This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the show was not really funny anymore, and

2. Filter blank text review. Two files are blank in this dataset.

In [27]:
import tensorflow as tf # Added for robustness

def filter_empty_reviews(text, label):
    return tf.strings.length(text) > 0

filtered_reviews_ds = processed_reviews_ds.filter(filter_empty_reviews)

print("\nFirst few processed and filtered reviews with labels:\n")
for text, label in filtered_reviews_ds.take(5):
    print(f"Text: {text.numpy().decode('utf-8')}, Label: {label.numpy()}")


First few processed and filtered reviews with labels:

Text: Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.
, Label: 0
Text: This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the sh


3. Do all of the above transformations in single line of code. Also shuffle all the reviews

In [28]:
import tensorflow as tf # Added for robustness

final_reviews_ds = tf.data.Dataset.list_files(os.path.join(base_path, '*/*'), shuffle=True) \
    .map(process_path) \
    .filter(filter_empty_reviews)

print("\nFirst few elements from the final shuffled, processed, and filtered dataset:")

for text, label in final_reviews_ds.take(5):
    print(f"Text: {text.numpy().decode('utf-8')}, Label: {label.numpy()}")


First few elements from the final shuffled, processed, and filtered dataset:
Text: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are 

In [33]:
import tensorflow as tf

final_review_ds = tf.data.Dataset.list_files(os.path.join(base_path, "*/*")).filter(lambda y : tf.strings.length(y) > 0).map(lambda x : (tf.io.read_file(x), tf.where(tf.equal(tf.strings.split(x, os.path.sep)[-2], "positive"), 1, 0))).shuffle(2).batch(2)

for text, label in final_review_ds.take(5):
    print(f"Text: {text.numpy()}, Label: {label.numpy()}")

Text: [b''
 b"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is 